# Graph Neural Network fingerprints for Data 1.1

## 1. Data ingestion and cleaning

In [ ]:
import pandas as pd

df_cdd = pd.read_csv('data_1.csv')
df_cdd = df_cdd.rename(columns={
    "GU 10 µM L6: Average Glucose uptake / isoprenaline (%)": "gu",
    "B2 cAMP L6: Average cAMP / isoprenaline (AU)": "camp",
})

In [ ]:
df_cdd = df_cdd.drop(['Synonyms',
                      'GU 10 µM L6: Average Glucose uptake / isoprenaline (%) Standard Deviation (±)',
                      'GU 10 µM L6: Average Glucose uptake / isoprenaline (%) Count',
                      'B2 cAMP L6: Average cAMP / isoprenaline (AU) Standard Deviation (±)',
                      'B2 cAMP L6: Average cAMP / isoprenaline (AU) Count'
                    ], axis=1)

### 1.1 — Install Dependencies

In [ ]:
# Run once; restart kernel after if needed
import subprocess, sys

pkgs = [
    'torch',
    'torch-geometric',
    'rdkit-pypi',
    'captum',
    'pandas',
    'numpy',
    'matplotlib',
    'seaborn',
    'scikit-learn',
    'tqdm',
]

# Install PyG extras separately (CPU-only example)
pyg_deps = [
    'torch-scatter',
    'torch-sparse',
    'torch-cluster',
]

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
print("Core packages installed.")
print("Note: If torch-geometric extras fail, install them manually following https://pytorch-geometric.readthedocs.io/en/latest/install/installation.html")

### 1.2 — Import libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from copy import deepcopy
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

import torch_geometric
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import NNConv, global_mean_pool, global_add_pool
from torch_geometric.explain import Explainer, GNNExplainer

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from rdkit import Chem
from rdkit.Chem import Draw, AllChem, rdMolDescriptors
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import display, Image

# For Integrated Gradients
from captum.attr import IntegratedGradients

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")
print(f"PyTorch Geometric version: {torch_geometric.__version__}")

### 1.3 — Dataset Loading

> The dataset must have columns: `Molecule Name`, `SMILES`, `gu`, `camp`.

In [ ]:
df = df_cdd
df = df.dropna(subset=['camp'])
df.columns = df.columns.str.strip()

df.head()

### 1.4 — EDA: Distribution of Targets

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(df['gu'], bins=15, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Glucose Uptake (GU)', fontsize=12)
axes[0].set_ylabel('Count')
axes[0].set_title('GU Distribution')

axes[1].hist(df['camp'], bins=15, color='coral', edgecolor='white')
axes[1].set_xlabel('cAMP', fontsize=12)
axes[1].set_ylabel('Count')
axes[1].set_title('cAMP Distribution')

axes[2].scatter(df['gu'], df['camp'], alpha=0.7, c='purple', edgecolors='k', linewidths=0.4)
axes[2].set_xlabel('GU (↑ desired)', fontsize=12)
axes[2].set_ylabel('cAMP (↓ desired)', fontsize=12)
axes[2].set_title('GU vs cAMP')
axes[2].axhline(df['camp'].median(), color='red', linestyle='--', alpha=0.5, label='cAMP median')
axes[2].axvline(df['gu'].median(), color='blue', linestyle='--', alpha=0.5, label='GU median')
axes[2].legend()

plt.suptitle('B2AR Dataset — Target Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_targets.png', dpi=150, bbox_inches='tight')
plt.show()
print("Correlation GU vs cAMP:", df[['gu','camp']].corr().iloc[0,1].round(3))

## 2. — Molecular Feature Engineering

In [ ]:
# ─── Atom Features ────────────────────────────────────────────────────────────
ATOM_TYPES = ['C','N','O','F','P','S','Cl','Br','I','Se','Si','Other']
HYBRIDIZATION = [
    Chem.rdchem.HybridizationType.SP,
    Chem.rdchem.HybridizationType.SP2,
    Chem.rdchem.HybridizationType.SP3,
    Chem.rdchem.HybridizationType.SP3D,
    Chem.rdchem.HybridizationType.SP3D2,
]

def one_hot(value, categories):
    vec = [0] * (len(categories) + 1)  # +1 for 'other'
    if value in categories:
        vec[categories.index(value)] = 1
    else:
        vec[-1] = 1
    return vec

def atom_features(atom):
    """18-dimensional atom feature vector."""
    return (
        one_hot(atom.GetSymbol(), ATOM_TYPES)                    # 13
        + one_hot(atom.GetHybridization(), HYBRIDIZATION)         # 6
        + [int(atom.GetIsAromatic())]                             # 1
        + [atom.GetTotalNumHs() / 4.0]                            # 1
        + [atom.GetFormalCharge() / 2.0]                          # 1
        + [int(atom.IsInRing())]                                  # 1
        # total = 23
    )

# ─── Bond Features ────────────────────────────────────────────────────────────
BOND_TYPES = [
    Chem.rdchem.BondType.SINGLE,
    Chem.rdchem.BondType.DOUBLE,
    Chem.rdchem.BondType.TRIPLE,
    Chem.rdchem.BondType.AROMATIC,
]

def bond_features(bond):
    """6-dimensional bond feature vector."""
    return (
        one_hot(bond.GetBondType(), BOND_TYPES)   # 5 (4+1 other)
        + [int(bond.IsInRing())]                  # 1
        # total = 6
    )

NODE_DIM = len(atom_features(Chem.MolFromSmiles('C').GetAtomWithIdx(0)))
EDGE_DIM = len(bond_features(Chem.MolFromSmiles('CC').GetBondWithIdx(0)))
print(f"Node feature dimension: {NODE_DIM}")
print(f"Edge feature dimension: {EDGE_DIM}")

### 2.1 — SMILES → PyG Graphs

In [ ]:
def smiles_to_graph(smiles, gu_val, camp_val, mol_name):
    """Convert SMILES string to PyG Data object."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # Node features
    x = torch.tensor(
        [atom_features(atom) for atom in mol.GetAtoms()],
        dtype=torch.float
    )

    # Edge index and features (bidirectional)
    edge_index, edge_attr = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = bond_features(bond)
        edge_index += [[i, j], [j, i]]
        edge_attr += [bf, bf]

    if len(edge_index) == 0:  # single atom molecules
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, EDGE_DIM), dtype=torch.float)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attr, dtype=torch.float)

    y = torch.tensor([[gu_val, camp_val]], dtype=torch.float)  # shape (1,2)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
    data.smiles = smiles
    data.name = mol_name
    return data


# Build dataset
dataset = []
failed = []
for _, row in df.iterrows():
    g = smiles_to_graph(row['SMILES'], row['gu'], row['camp'], row['Molecule Name'])
    if g is not None:
        dataset.append(g)
    else:
        failed.append(row['Molecule Name'])

print(f"Successfully converted: {len(dataset)} / {len(df)} molecules")
if failed:
    print(f"Failed (invalid SMILES): {failed}")

# Quick sanity check
g = dataset[0]
print(f"\nExample graph — {g.name}")
print(f"  Nodes: {g.num_nodes}, Node features: {g.x.shape[1]}")
print(f"  Edges: {g.num_edges}, Edge features: {g.edge_attr.shape[1]}")
print(f"  Target y: {g.y}  (GU, cAMP)")

### 2.2 — Target Normalization

In [ ]:
# Fit scaler on full dataset targets
Y_raw = np.array([g.y.numpy().flatten() for g in dataset])  # (N, 2)
scaler = StandardScaler()
Y_scaled = scaler.fit_transform(Y_raw)

# Update graph objects with scaled targets
dataset_scaled = []
for i, g in enumerate(dataset):
    g2 = deepcopy(g)
    g2.y = torch.tensor(Y_scaled[i:i+1], dtype=torch.float)  # (1,2)
    g2.y_raw = torch.tensor(Y_raw[i:i+1], dtype=torch.float)  # keep original
    dataset_scaled.append(g2)

print(f"Target scaler means: GU={scaler.mean_[0]:.3f}, cAMP={scaler.mean_[1]:.3f}")
print(f"Target scaler stds : GU={scaler.scale_[0]:.3f}, cAMP={scaler.scale_[1]:.3f}")
print("Scaled Y sample:")
print(Y_scaled[:3])

### 2.3 — Multi-Output MPNN Architecture

In [ ]:
class MultiOutputMPNN(nn.Module):
    """
    Message Passing Neural Network with:
    - NNConv (edge-conditioned convolution) for 2 MP layers
    - Global mean pooling
    - Shared MLP trunk
    - Two output heads: GU and cAMP (multi-output, NOT separate models)
    """

    def __init__(self, node_dim, edge_dim, hidden_dim=64, mp_layers=2, dropout=0.2):
        super().__init__()
        self.node_dim = node_dim
        self.edge_dim = edge_dim
        self.hidden_dim = hidden_dim

        # Initial node embedding
        self.node_emb = nn.Sequential(
            nn.Linear(node_dim, hidden_dim),
            nn.ReLU(),
        )

        # NNConv layers — edge network maps edge features to weight matrices
        self.conv_layers = nn.ModuleList()
        self.bn_layers = nn.ModuleList()
        for _ in range(mp_layers):
            edge_net = nn.Sequential(
                nn.Linear(edge_dim, hidden_dim * hidden_dim),
                nn.ReLU(),
            )
            self.conv_layers.append(NNConv(hidden_dim, hidden_dim, edge_net, aggr='mean'))
            self.bn_layers.append(nn.BatchNorm1d(hidden_dim))

        self.dropout = nn.Dropout(dropout)

        # Shared trunk
        self.trunk = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
        )

        # ── Two output heads (multi-output) ──────────────────────────────────
        self.head_gu   = nn.Linear(hidden_dim // 2, 1)   # Glucose Uptake head
        self.head_camp = nn.Linear(hidden_dim // 2, 1)   # cAMP head

    def forward(self, x, edge_index, edge_attr, batch):
        # Initial embedding
        h = self.node_emb(x)

        # Message passing
        for conv, bn in zip(self.conv_layers, self.bn_layers):
            h = conv(h, edge_index, edge_attr)
            h = bn(h)
            h = F.relu(h)
            h = self.dropout(h)

        # Graph-level readout
        h = global_mean_pool(h, batch)   # (batch_size, hidden_dim)

        # Shared trunk
        h = self.trunk(h)

        # Separate heads
        out_gu   = self.head_gu(h)    # (batch_size, 1)
        out_camp = self.head_camp(h)  # (batch_size, 1)

        return torch.cat([out_gu, out_camp], dim=1)  # (batch_size, 2)


# Test instantiation
model_test = MultiOutputMPNN(node_dim=NODE_DIM, edge_dim=EDGE_DIM)
print(model_test)
total_params = sum(p.numel() for p in model_test.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params:,}")

### 2.4 — Training & Evaluation Utilities

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        pred = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        loss = criterion(pred, batch.y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    preds, targets = [], []
    for batch in loader:
        batch = batch.to(device)
        pred = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        loss = criterion(pred, batch.y)
        total_loss += loss.item() * batch.num_graphs
        preds.append(pred.cpu())
        targets.append(batch.y.cpu())
    preds   = torch.cat(preds).numpy()
    targets = torch.cat(targets).numpy()
    return total_loss / len(loader.dataset), preds, targets


def compute_metrics(preds_scaled, targets_scaled, scaler):
    """Return metrics in original (unscaled) space."""
    preds_orig   = scaler.inverse_transform(preds_scaled)
    targets_orig = scaler.inverse_transform(targets_scaled)
    metrics = {}
    for i, name in enumerate(['GU', 'cAMP']):
        metrics[f'R2_{name}']   = r2_score(targets_orig[:, i], preds_orig[:, i])
        metrics[f'MAE_{name}']  = mean_absolute_error(targets_orig[:, i], preds_orig[:, i])
        metrics[f'RMSE_{name}'] = np.sqrt(mean_squared_error(targets_orig[:, i], preds_orig[:, i]))
    return metrics, preds_orig, targets_orig

print("Training utilities defined.")

## 3. 5-Fold Cross-Validation

In [ ]:
# ─── Hyperparameters ──────────────────────────────────────────────────────────
EPOCHS     = 150
LR         = 1e-3
BATCH_SIZE = 16
HIDDEN_DIM = 64
N_FOLDS    = 5
PATIENCE   = 25   # early stopping patience

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
indices = list(range(len(dataset_scaled)))

fold_metrics   = []
fold_models    = []   # keep best model per fold for XAI
all_preds_orig = np.zeros((len(dataset_scaled), 2))
all_true_orig  = np.zeros((len(dataset_scaled), 2))
train_loss_history = []
val_loss_history   = []

criterion = nn.MSELoss()

for fold, (train_idx, val_idx) in enumerate(kf.split(indices)):
    print(f"\n{'='*50}")
    print(f" Fold {fold+1}/{N_FOLDS}  |  Train: {len(train_idx)}  |  Val: {len(val_idx)}")
    print(f"{'='*50}")

    train_data = [dataset_scaled[i] for i in train_idx]
    val_data   = [dataset_scaled[i] for i in val_idx]

    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_data,   batch_size=BATCH_SIZE, shuffle=False)

    model = MultiOutputMPNN(
        node_dim=NODE_DIM, edge_dim=EDGE_DIM, hidden_dim=HIDDEN_DIM
    ).to(DEVICE)

    optimizer = Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = ReduceLROnPlateau(optimizer, patience=10, factor=0.5)

    best_val_loss  = float('inf')
    best_state     = None
    patience_count = 0
    fold_train_losses = []
    fold_val_losses   = []

    for epoch in range(1, EPOCHS + 1):
        tr_loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
        va_loss, va_preds, va_targets = evaluate(model, val_loader, criterion, DEVICE)
        scheduler.step(va_loss)

        fold_train_losses.append(tr_loss)
        fold_val_losses.append(va_loss)

        if va_loss < best_val_loss:
            best_val_loss  = va_loss
            best_state     = deepcopy(model.state_dict())
            patience_count = 0
        else:
            patience_count += 1

        if patience_count >= PATIENCE:
            print(f"  Early stopping at epoch {epoch}")
            break

        if epoch % 30 == 0:
            print(f"  Epoch {epoch:3d} | Train Loss: {tr_loss:.4f} | Val Loss: {va_loss:.4f}")

    # Restore best weights
    model.load_state_dict(best_state)
    fold_models.append(deepcopy(model))

    # Final evaluation on validation fold
    _, val_preds_sc, val_targets_sc = evaluate(model, val_loader, criterion, DEVICE)
    metrics, preds_orig, targets_orig = compute_metrics(val_preds_sc, val_targets_sc, scaler)

    # Store predictions for later (oof)
    for k, idx in enumerate(val_idx):
        all_preds_orig[idx] = preds_orig[k]
        all_true_orig[idx]  = targets_orig[k]

    fold_metrics.append(metrics)
    train_loss_history.append(fold_train_losses)
    val_loss_history.append(fold_val_losses)

    print(f"\n  Best val MSE (scaled): {best_val_loss:.4f}")
    print(f"  GU   — R²: {metrics['R2_GU']:.3f} | MAE: {metrics['MAE_GU']:.3f} | RMSE: {metrics['RMSE_GU']:.3f}")
    print(f"  cAMP — R²: {metrics['R2_cAMP']:.3f} | MAE: {metrics['MAE_cAMP']:.3f} | RMSE: {metrics['RMSE_cAMP']:.3f}")

print("\n5-fold cross-validation complete.")

### 3.1 — Cross-Validation Summary

In [ ]:
metrics_df = pd.DataFrame(fold_metrics)
metrics_df.index = [f'Fold {i+1}' for i in range(N_FOLDS)]

summary = metrics_df.agg(['mean','std'])
print("\n=== 5-Fold Cross-Validation Results ===")
print(metrics_df.round(3).to_string())
print("\n=== Summary ===")
print(summary.round(3).to_string())

# ─── Bar chart of R² across folds ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, target, color in zip(axes, ['GU','cAMP'], ['steelblue','coral']):
    vals = metrics_df[f'R2_{target}']
    ax.bar(range(1, N_FOLDS+1), vals, color=color, alpha=0.8, edgecolor='k')
    ax.axhline(vals.mean(), color='black', linestyle='--', label=f'Mean={vals.mean():.3f}')
    ax.set_xlabel('Fold'); ax.set_ylabel('R²')
    ax.set_title(f'{target} R² per Fold')
    ax.set_xticks(range(1, N_FOLDS+1))
    ax.legend()
plt.tight_layout()
plt.savefig('cv_r2.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Model Training
### 4.1 — Training Curve Visualization

In [ ]:
fig, axes = plt.subplots(1, N_FOLDS, figsize=(18, 3), sharey=False)
for fold_i, ax in enumerate(axes):
    ax.plot(train_loss_history[fold_i], label='Train', color='steelblue')
    ax.plot(val_loss_history[fold_i],   label='Val',   color='coral')
    ax.set_title(f'Fold {fold_i+1}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss (scaled)')
    ax.legend(fontsize=8)

plt.suptitle('Training Curves — 5-Fold CV', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.2 — OOF Predicted vs Actual Scatter

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (i, target, color) in zip(axes, [(0,'GU','steelblue'), (1,'cAMP','coral')]):
    ax.scatter(all_true_orig[:, i], all_preds_orig[:, i],
               alpha=0.7, c=color, edgecolors='k', linewidths=0.4, s=60)
    lims = [min(all_true_orig[:,i].min(), all_preds_orig[:,i].min()),
            max(all_true_orig[:,i].max(), all_preds_orig[:,i].max())]
    ax.plot(lims, lims, 'k--', alpha=0.5, label='Ideal')
    r2  = r2_score(all_true_orig[:,i], all_preds_orig[:,i])
    mae = mean_absolute_error(all_true_orig[:,i], all_preds_orig[:,i])
    ax.set_title(f'{target} — OOF  |  R²={r2:.3f}  |  MAE={mae:.3f}')
    ax.set_xlabel(f'True {target}')
    ax.set_ylabel(f'Predicted {target}')
    ax.legend()

plt.suptitle('Out-of-Fold Predictions vs Actual', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('oof_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. — Bias Score: Capturing High-GU + Low-cAMP

The **Bias Score** quantifies how strongly each molecule sits in the "desirable" region:  
> High GU AND Low cAMP simultaneously.

$$\text{BiasScore} = \frac{\hat{GU}_{norm} + (1 - \hat{cAMP}_{norm})}{2} \in [0, 1]$$

where normalized values are min-max scaled across the full dataset.

In [ ]:
def compute_bias_score(gu_vals, camp_vals):
    """
    Bias Score: combines high GU (desirable) and low cAMP (desirable).
    Returns values in [0, 1] where 1 = perfectly desirable.
    """
    gu_arr   = np.array(gu_vals, dtype=float)
    camp_arr = np.array(camp_vals, dtype=float)

    # Min-max normalize
    def minmax(x):
        rng = x.max() - x.min()
        if rng < 1e-9:
            return np.zeros_like(x)
        return (x - x.min()) / rng

    gu_norm   = minmax(gu_arr)          # 1 = high (good)
    camp_norm = 1.0 - minmax(camp_arr)  # 1 = low cAMP (good)

    return (gu_norm + camp_norm) / 2.0


# Compute bias score using OOF predictions (model ensemble output)
bias_scores = compute_bias_score(all_preds_orig[:, 0], all_preds_orig[:, 1])

# Also compute for true values
bias_scores_true = compute_bias_score(all_true_orig[:, 0], all_true_orig[:, 1])

# Attach to molecules
results_df = pd.DataFrame({
    'Molecule Name': [dataset_scaled[i].name for i in range(len(dataset_scaled))],
    'SMILES':        [dataset_scaled[i].smiles for i in range(len(dataset_scaled))],
    'GU_true':       all_true_orig[:, 0],
    'cAMP_true':     all_true_orig[:, 1],
    'GU_pred':       all_preds_orig[:, 0],
    'cAMP_pred':     all_preds_orig[:, 1],
    'BiasScore_pred': bias_scores,
    'BiasScore_true': bias_scores_true,
})
results_df = results_df.sort_values('BiasScore_pred', ascending=False).reset_index(drop=True)

print("=== Top-10 Molecules by Bias Score (Predicted High-GU & Low-cAMP) ===")
print(results_df[['Molecule Name','GU_pred','cAMP_pred','BiasScore_pred']].head(10).round(3).to_string())

# Plot
fig, ax = plt.subplots(figsize=(10, 4))
cmap = cm.RdYlGn
sc = ax.scatter(
    results_df['GU_pred'], results_df['cAMP_pred'],
    c=results_df['BiasScore_pred'], cmap=cmap, s=80,
    edgecolors='k', linewidths=0.4, vmin=0, vmax=1
)
plt.colorbar(sc, ax=ax, label='Bias Score (↑ better)')
ax.set_xlabel('Predicted GU (↑ desired)', fontsize=12)
ax.set_ylabel('Predicted cAMP (↓ desired)', fontsize=12)
ax.set_title('Bias Score Map: High-GU & Low-cAMP Region (Green = Desirable)', fontsize=13)

# Annotate top-3
for _, row in results_df.head(3).iterrows():
    ax.annotate(row['Molecule Name'], (row['GU_pred'], row['cAMP_pred']),
                fontsize=8, ha='left', xytext=(5, 5), textcoords='offset points')

plt.tight_layout()
plt.savefig('bias_score_map.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.1 — Final Ensemble Model (Trained on Full Dataset)

For XAI, we train one final model on the **entire dataset** (all 5-fold models already exist — here we also train a single full-dataset model for cleaner XAI attribution).

In [ ]:
full_loader = DataLoader(dataset_scaled, batch_size=BATCH_SIZE, shuffle=True)

final_model = MultiOutputMPNN(
    node_dim=NODE_DIM, edge_dim=EDGE_DIM, hidden_dim=HIDDEN_DIM
).to(DEVICE)

optimizer_final = Adam(final_model.parameters(), lr=LR, weight_decay=1e-4)
scheduler_final = ReduceLROnPlateau(optimizer_final, patience=15, factor=0.5)

best_loss_final = float('inf')
best_state_final = None
patience_cnt = 0
EPOCHS_FINAL = 200

print("Training final model on full dataset...")
for epoch in range(1, EPOCHS_FINAL + 1):
    loss = train_epoch(final_model, full_loader, optimizer_final, criterion, DEVICE)
    scheduler_final.step(loss)
    if loss < best_loss_final:
        best_loss_final = loss
        best_state_final = deepcopy(final_model.state_dict())
        patience_cnt = 0
    else:
        patience_cnt += 1
    if patience_cnt >= 30:
        print(f"  Early stopping at epoch {epoch}")
        break
    if epoch % 50 == 0:
        print(f"  Epoch {epoch} | Loss: {loss:.4f}")

final_model.load_state_dict(best_state_final)
final_model.eval()
print(f"Final model trained. Best loss: {best_loss_final:.4f}")

## 6. — XAI Setup: Target Selection for Combined Desirability

For XAI we need a **single scalar objective** that captures high-GU + low-cAMP.  
We create a wrapper model that outputs the **Bias Score** as a differentiable function.

In [ ]:
class BiasScoreModel(nn.Module):
    """
    Wraps the trained MPNN and computes a differentiable Bias Score:
        BiasScore = sigmoid(GU_pred) - sigmoid(cAMP_pred)
    Higher = more desirable (high GU, low cAMP).
    
    The sigmoid maps predictions to (0,1), so:
    - We want to maximize sigmoid(GU) → maximize GU
    - We want to minimize sigmoid(cAMP) → minimize cAMP
    """
    def __init__(self, base_model):
        super().__init__()
        self.base = base_model

    def forward(self, x, edge_index, edge_attr, batch):
        out = self.base(x, edge_index, edge_attr, batch)  # (B, 2)
        gu_scaled   = out[:, 0]   # GU (higher=better after inversion of scaler)
        camp_scaled = out[:, 1]   # cAMP (lower=better)
        # Differentiable proxy for combined desirability
        score = torch.sigmoid(gu_scaled) - torch.sigmoid(camp_scaled)  # (B,)
        return score


bias_model = BiasScoreModel(final_model).to(DEVICE)
bias_model.eval()

# Quick test
g0 = dataset_scaled[0].to(DEVICE)
with torch.no_grad():
    dummy_batch = torch.zeros(g0.num_nodes, dtype=torch.long, device=DEVICE)
    bs = bias_model(g0.x, g0.edge_index, g0.edge_attr, dummy_batch)
print(f"BiasScore for {dataset_scaled[0].name}: {bs.item():.4f}")
print("  (Range ≈ -1 to +1; higher = better)")

### 6.1 — GNNExplainer (Stochasticity-Corrected)

GNNExplainer is stochastic — run **N_RUNS times per molecule** and aggregate node importance by mean, accounting for variance.

In [ ]:
from torch_geometric.explain import Explainer, GNNExplainer, ExplainerConfig, ModelConfig

N_RUNS = 10       # number of GNNExplainer runs per molecule (to average stochasticity)
GNN_EPOCHS = 200  # GNNExplainer optimization epochs

def run_gnnexplainer_multi(
    model, data, device, n_runs=10, epochs=200, node_mask_type='attributes'
):
    """
    Run GNNExplainer n_runs times and return:
    - mean_node_imp: averaged node importance across runs (shape: [n_nodes])
    - std_node_imp:  std of node importance (stochasticity measure)
    """
    all_node_masks = []
    x          = data.x.to(device)
    edge_index = data.edge_index.to(device)
    edge_attr  = data.edge_attr.to(device)
    batch      = torch.zeros(x.size(0), dtype=torch.long, device=device)

    for run in range(n_runs):
        explainer = Explainer(
            model=model,
            algorithm=GNNExplainer(epochs=epochs),
            explanation_type='model',
            node_mask_type='attributes',
            edge_mask_type='object',
            model_config=ModelConfig(
                mode='regression',
                task_level='graph',
                return_type='raw',
            ),
        )
        explanation = explainer(
            x=x,
            edge_index=edge_index,
            edge_attr=edge_attr,
            batch=batch,
            index=None
        )
        # Node mask: (n_nodes, n_features) → aggregate to per-node importance
        node_mask = explanation.node_mask.detach().cpu()   # (n_nodes, node_dim)
        node_imp  = node_mask.abs().mean(dim=1)            # (n_nodes,)
        all_node_masks.append(node_imp.numpy())

    all_node_masks = np.stack(all_node_masks)  # (n_runs, n_nodes)
    return all_node_masks.mean(axis=0), all_node_masks.std(axis=0)


# ─── Run GNNExplainer on full dataset ─────────────────────────────────────────
print(f"Running GNNExplainer ({N_RUNS} runs × {len(dataset_scaled)} molecules)...")
gnn_node_imps_mean = []  # list of arrays, one per molecule
gnn_node_imps_std  = []

for idx, g in enumerate(tqdm(dataset_scaled, desc='GNNExplainer')):
    mean_imp, std_imp = run_gnnexplainer_multi(
        bias_model, g, DEVICE, n_runs=N_RUNS, epochs=GNN_EPOCHS
    )
    gnn_node_imps_mean.append(mean_imp)
    gnn_node_imps_std.append(std_imp)

print("GNNExplainer complete.")

### 6.2 — GNNExplainer: Aggregate Atom-Type Importance

In [ ]:
# For each molecule, weight node importance by its bias score
# → nodes in high-scoring molecules contribute more
bias_arr = np.array(results_df.set_index('Molecule Name').loc[
    [dataset_scaled[i].name for i in range(len(dataset_scaled))], 'BiasScore_pred'
])

# Aggregate atom-type importances across ALL molecules
atom_type_importance_gnn = {at: [] for at in ATOM_TYPES}

for i, g in enumerate(dataset_scaled):
    mol = Chem.MolFromSmiles(g.smiles)
    if mol is None:
        continue
    imp = gnn_node_imps_mean[i]  # (n_atoms,)
    w   = bias_arr[i]            # bias weight for this molecule
    for atom_idx, atom in enumerate(mol.GetAtoms()):
        sym = atom.GetSymbol() if atom.GetSymbol() in ATOM_TYPES else 'Other'
        atom_type_importance_gnn[sym].append(imp[atom_idx] * w)

# Mean importance per atom type
atom_means_gnn = {k: np.mean(v) if v else 0 for k, v in atom_type_importance_gnn.items()}
atom_stds_gnn  = {k: np.std(v)  if v else 0 for k, v in atom_type_importance_gnn.items()}

# Plot
atoms_sorted = sorted(atom_means_gnn, key=atom_means_gnn.get, reverse=True)
means = [atom_means_gnn[a] for a in atoms_sorted]
stds  = [atom_stds_gnn[a]  for a in atoms_sorted]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(atoms_sorted, means, yerr=stds, capsize=4,
              color='mediumseagreen', edgecolor='k', alpha=0.85, error_kw={'linewidth': 1.5})
ax.set_xlabel('Atom Type', fontsize=12)
ax.set_ylabel('Mean Bias-Weighted Node Importance', fontsize=11)
ax.set_title('GNNExplainer — Atom-Type Importance (High-GU & Low-cAMP Objective)\n'
             f'(Averaged over {N_RUNS} stochastic runs)', fontsize=12)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('gnnexplainer_atom_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nAtom type importances (GNNExplainer):")
for a in atoms_sorted:
    print(f"  {a:6s}: {atom_means_gnn[a]:.4f} ± {atom_stds_gnn[a]:.4f}")

### 6.3 — GNNExplainer: Molecule Visualization with Atom Importance

In [ ]:
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import SVG

def draw_mol_with_weights(smiles, atom_weights, title='', size=(400, 300)):
    """Draw molecule with per-atom importance as color intensity."""
    mol = Chem.MolFromSmiles(smiles)
    AllChem.Compute2DCoords(mol)

    w = np.array(atom_weights)
    if w.max() > 0:
        w = w / w.max()  # normalize to [0,1]

    # Green colormap for importance
    atom_colors = {}
    for i, val in enumerate(w):
        atom_colors[i] = (1 - val * 0.8, 1.0, 1 - val * 0.8)  # light→dark green

    drawer = rdMolDraw2D.MolDraw2DSVG(size[0], size[1])
    drawer.drawOptions().addAtomIndices = False
    rdMolDraw2D.PrepareAndDrawMolecule(
        drawer, mol,
        highlightAtoms=list(range(mol.GetNumAtoms())),
        highlightAtomColors=atom_colors,
        highlightBonds=[],
    )
    drawer.DrawMolecule(mol,
                        highlightAtoms=list(range(mol.GetNumAtoms())),
                        highlightAtomColors=atom_colors,
                        highlightBonds=[])
    drawer.FinishDrawing()
    return drawer.GetDrawingText()


# Visualize top-3 molecules by bias score
top3_names = results_df['Molecule Name'].head(3).tolist()
top3_idx   = [i for i, g in enumerate(dataset_scaled) if g.name in top3_names]

fig, axes = plt.subplots(1, len(top3_idx), figsize=(14, 4))
for k, (idx, ax) in enumerate(zip(top3_idx, axes)):
    g = dataset_scaled[idx]
    mol = Chem.MolFromSmiles(g.smiles)
    AllChem.Compute2DCoords(mol)

    imp = gnn_node_imps_mean[idx]
    if imp.max() > 0:
        imp_norm = imp / imp.max()
    else:
        imp_norm = imp

    # Plot using matplotlib atom scatter overlay approach
    from rdkit.Chem import rdDepictor
    from rdkit.Chem.Draw import MolToImage

    conf = mol.GetConformer()
    positions = conf.GetPositions()
    xs = positions[:, 0]
    ys = positions[:, 1]

    img = Draw.MolToImage(mol, size=(300, 220))
    ax.imshow(img)
    ax.axis('off')

    bs = results_df.loc[results_df['Molecule Name']==g.name, 'BiasScore_pred'].values[0]
    ax.set_title(f"{g.name}\nBias={bs:.3f} | Nodes: {g.num_nodes}", fontsize=9)

plt.suptitle('Top-3 Molecules by Bias Score (GNNExplainer)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('top3_gnnexplainer.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from IPython.display import SVG, display

def visualize_molecule_by_name(molecule_name):
    # 1️⃣ Get row from results_df
    row = results_df.loc[results_df['Molecule Name'] == molecule_name]
    
    if row.empty:
        print(f"Molecule '{molecule_name}' not found.")
        return
    
    bias_score = row['BiasScore_pred'].values[0]

    # 2️⃣ Find index in dataset_scaled
    idx = None
    for i, g in enumerate(dataset_scaled):
        if g.name == molecule_name:
            idx = i
            break
    
    if idx is None:
        print(f"Molecule '{molecule_name}' not found in dataset.")
        return

    g = dataset_scaled[idx]

    # 3️⃣ Get importance scores
    imp = gnn_node_imps_mean[idx]

    # 4️⃣ Draw molecule with importance coloring
    svg = draw_mol_with_weights(
        smiles=g.smiles,
        atom_weights=imp,
        title=f"{molecule_name} | Bias={bias_score:.3f}"
    )

    # 5️⃣ Display
    display(SVG(svg))

### 6.4 — Integrated Gradients (Captum)

Integrated Gradients attributes the model output to **node features**, providing feature-level importance.
We attribute with respect to the `BiasScore` output.

In [ ]:
class IGWrapper(nn.Module):
    """
    Wrapper for Captum Integrated Gradients.
    Captum requires a forward(x, ...) signature where x is the tensor to attribute.
    We pass edge_index, edge_attr, batch as fixed additional args.
    """
    def __init__(self, bias_model):
        super().__init__()
        self.bias_model = bias_model

    def forward(self, x, edge_index, edge_attr, batch):
        out = self.bias_model(x, edge_index, edge_attr, batch)  # (batch_size,)
        return out.sum().unsqueeze(0)   # scalar for IG


ig_wrapper = IGWrapper(bias_model).to(DEVICE)
ig_wrapper.eval()
ig = IntegratedGradients(ig_wrapper)


def compute_ig_attributions(g, device, n_steps=50):
    """Compute IG attributions for a single graph."""
    x          = g.x.to(device).requires_grad_(True).float()
    edge_index = g.edge_index.to(device)
    edge_attr  = g.edge_attr.to(device)
    batch      = torch.zeros(x.size(0), dtype=torch.long, device=device)

    # Baseline: zero node features
    baseline = torch.zeros_like(x)

    attributions = ig.attribute(
        x,
        baselines=baseline,
        additional_forward_args=(edge_index, edge_attr, batch),
        n_steps=n_steps,
        internal_batch_size=x.size(0)
    )  # (n_nodes, node_dim)

    return attributions.detach().cpu().numpy()


# ─── Compute IG for all molecules ─────────────────────────────────────────────
print("Computing Integrated Gradients for all molecules...")
ig_attributions_all = []

for g in tqdm(dataset_scaled, desc='Integrated Gradients'):
    attrs = compute_ig_attributions(g, DEVICE)
    ig_attributions_all.append(attrs)  # (n_nodes, node_dim)

print("Integrated Gradients complete.")

### 6.5 — IG: Feature-Level Importance Analysis

In [ ]:
# Build feature names
feature_names = (
    [f'AtomType_{at}' for at in ATOM_TYPES] + ['AtomType_Other']
    + [f'Hyb_{h}' for h in ['SP','SP2','SP3','SP3D','SP3D2']] + ['Hyb_Other']
    + ['IsAromatic', 'NumHs', 'FormalCharge', 'IsInRing']
)
assert len(feature_names) == NODE_DIM, f"{len(feature_names)} vs {NODE_DIM}"

# Aggregate IG attributions weighted by bias score
ig_feature_importance = np.zeros(NODE_DIM)
total_weight = 0.0

for i, (g, attrs) in enumerate(zip(dataset_scaled, ig_attributions_all)):
    bs = bias_arr[i]
    # Per-node absolute attribution, then mean over nodes → per-feature
    node_attr_abs = np.abs(attrs)           # (n_nodes, node_dim)
    mol_feat_imp  = node_attr_abs.mean(0)   # (node_dim,)
    ig_feature_importance += mol_feat_imp * bs
    total_weight += bs

ig_feature_importance /= (total_weight + 1e-9)

# Sort and display
sorted_idx = np.argsort(ig_feature_importance)[::-1]
top_n = 15

fig, ax = plt.subplots(figsize=(11, 5))
top_names = [feature_names[i] for i in sorted_idx[:top_n]]
top_vals  = [ig_feature_importance[i] for i in sorted_idx[:top_n]]
colors = ['#2196F3' if 'AtomType' in n else '#FF9800' if 'Hyb' in n else '#4CAF50'
          for n in top_names]
ax.barh(top_names[::-1], top_vals[::-1], color=colors[::-1], edgecolor='k', alpha=0.85)
ax.set_xlabel('Mean Bias-Weighted IG Attribution', fontsize=11)
ax.set_title(f'Integrated Gradients — Top-{top_n} Node Features\n'
             '(Blue=AtomType, Orange=Hybridization, Green=Other)\n'
             'Target: High GU & Low cAMP Bias Score', fontsize=12)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('ig_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop-15 Features (Integrated Gradients):")
for rank, i in enumerate(sorted_idx[:15]):
    print(f"  {rank+1:2d}. {feature_names[i]:<25s}: {ig_feature_importance[i]:.5f}")

### 6.7 — IG: Per-Atom Attribution on Top Molecules

In [ ]:
# Show per-atom IG attribution for top-3 bias score molecules
fig, axes = plt.subplots(1, min(3, len(top3_idx)), figsize=(14, 4))
if len(top3_idx) == 1:
    axes = [axes]

for k, (idx, ax) in enumerate(zip(top3_idx[:3], axes)):
    g    = dataset_scaled[idx]
    attrs = ig_attributions_all[idx]          # (n_nodes, node_dim)
    node_imp = np.abs(attrs).mean(axis=1)     # (n_nodes,)

    mol = Chem.MolFromSmiles(g.smiles)
    AllChem.Compute2DCoords(mol)
    img = Draw.MolToImage(mol, size=(300, 220))
    ax.imshow(img)
    ax.axis('off')

    bs = results_df.loc[results_df['Molecule Name']==g.name, 'BiasScore_pred'].values[0]
    ax.set_title(f"{g.name}\nBias={bs:.3f} | Nodes={g.num_nodes}", fontsize=9)

plt.suptitle('Top-3 Molecules — Integrated Gradients Attribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('top3_ig.png', dpi=150, bbox_inches='tight')
plt.show()

# Print per-atom IG for top molecule
idx0 = top3_idx[0]
g0   = dataset_scaled[idx0]
mol0 = Chem.MolFromSmiles(g0.smiles)
attrs0   = ig_attributions_all[idx0]
node_imp0 = np.abs(attrs0).mean(axis=1)

print(f"\nPer-Atom IG Attribution — {g0.name} (BiasScore={bias_arr[idx0]:.3f}):")
print(f"  {'Atom':^8} {'Symbol':^8} {'IG Importance':^15}")
print(f"  {'-'*35}")
for i, atom in enumerate(mol0.GetAtoms()):
    print(f"  {i:^8d} {atom.GetSymbol():^8s} {node_imp0[i]:^15.5f}")

## 7. — Comparison: GNNExplainer vs Integrated Gradients (Atom Types)

In [ ]:
# Compute IG atom-type importances (same breakdown as GNNExplainer)
atom_type_importance_ig = {at: [] for at in ATOM_TYPES}

for i, g in enumerate(dataset_scaled):
    mol = Chem.MolFromSmiles(g.smiles)
    if mol is None:
        continue
    attrs    = ig_attributions_all[i]         # (n_nodes, node_dim)
    node_imp = np.abs(attrs).mean(axis=1)     # (n_nodes,)
    w        = bias_arr[i]
    for atom_idx, atom in enumerate(mol.GetAtoms()):
        sym = atom.GetSymbol() if atom.GetSymbol() in ATOM_TYPES else 'Other'
        atom_type_importance_ig[sym].append(node_imp[atom_idx] * w)

atom_means_ig = {k: np.mean(v) if v else 0 for k, v in atom_type_importance_ig.items()}

# Side-by-side comparison
common_atoms = [a for a in ATOM_TYPES if atom_means_gnn.get(a,0) > 0 or atom_means_ig.get(a,0) > 0]
x_pos = np.arange(len(common_atoms))
width = 0.38

fig, ax = plt.subplots(figsize=(12, 5))
gnn_vals = [atom_means_gnn.get(a,0) for a in common_atoms]
ig_vals  = [atom_means_ig.get(a,0)  for a in common_atoms]

# Normalize each method to [0,1] for fair comparison
gnn_max = max(gnn_vals) or 1
ig_max  = max(ig_vals)  or 1

b1 = ax.bar(x_pos - width/2, [v/gnn_max for v in gnn_vals], width,
            label='GNNExplainer (normalized)', color='mediumseagreen', edgecolor='k', alpha=0.85)
b2 = ax.bar(x_pos + width/2, [v/ig_max  for v in ig_vals],  width,
            label='Integrated Gradients (normalized)', color='steelblue', edgecolor='k', alpha=0.85)

ax.set_xticks(x_pos)
ax.set_xticklabels(common_atoms, fontsize=11)
ax.set_ylabel('Normalized Bias-Weighted Importance', fontsize=11)
ax.set_title('GNNExplainer vs Integrated Gradients\n'
             'Atom-Type Importance for High-GU & Low-cAMP', fontsize=12)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('gnn_vs_ig_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
cmp_df = pd.DataFrame({
    'Atom': common_atoms,
    'GNNExplainer_norm': [v/gnn_max for v in gnn_vals],
    'IG_norm':           [v/ig_max  for v in ig_vals],
})
cmp_df = cmp_df.sort_values('GNNExplainer_norm', ascending=False)
print("\nAtom-type importance comparison (normalized):")
print(cmp_df.round(4).to_string(index=False))

### 7.1 — GNNExplainer Stochasticity Analysis

In [ ]:
# Coefficient of Variation (CV = std/mean) measures stochasticity per molecule
cv_per_mol = []
for i, (mean_imp, std_imp) in enumerate(zip(gnn_node_imps_mean, gnn_node_imps_std)):
    with np.errstate(divide='ignore', invalid='ignore'):
        cv = np.where(mean_imp > 1e-9, std_imp / mean_imp, 0)
    cv_per_mol.append(cv.mean())

cv_arr = np.array(cv_per_mol)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(cv_arr, bins=15, color='orange', edgecolor='k', alpha=0.85)
axes[0].axvline(cv_arr.mean(), color='red', linestyle='--',
                label=f'Mean CV = {cv_arr.mean():.3f}')
axes[0].set_xlabel('Coefficient of Variation (std/mean)')
axes[0].set_ylabel('Count')
axes[0].set_title(f'GNNExplainer Stochasticity ({N_RUNS} runs)\nLower = More Stable')
axes[0].legend()

axes[1].scatter(bias_arr, cv_arr, c='purple', alpha=0.7, edgecolors='k', linewidths=0.4)
axes[1].set_xlabel('Bias Score')
axes[1].set_ylabel('Mean CV (stochasticity)')
axes[1].set_title('Stochasticity vs Bias Score\n(Are high-scoring molecules more stable?)')

corr = np.corrcoef(bias_arr, cv_arr)[0,1]
axes[1].text(0.05, 0.95, f'r = {corr:.3f}', transform=axes[1].transAxes,
             fontsize=11, va='top')

plt.suptitle('GNNExplainer Stochasticity Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('gnnexplainer_stochasticity.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Mean CV across molecules: {cv_arr.mean():.4f} ± {cv_arr.std():.4f}")
print(f"Correlation (Bias Score vs CV): {corr:.4f}")
print(f"\nMolecules with highest explanation stability (lowest CV):")
stable_idx = np.argsort(cv_arr)[:5]
for si in stable_idx:
    print(f"  {dataset_scaled[si].name}: CV={cv_arr[si]:.4f}, BiasScore={bias_arr[si]:.3f}")

### 7.2 — Structural Pattern Summary: Substructure Frequency in High-Scoring Molecules

In [ ]:
from rdkit.Chem import Fragments

# Compare top-25% vs bottom-25% molecules by bias score
q75 = np.percentile(bias_arr, 75)
q25 = np.percentile(bias_arr, 25)

high_idx = [i for i, b in enumerate(bias_arr) if b >= q75]
low_idx  = [i for i, b in enumerate(bias_arr) if b <= q25]

# Common functional group SMARTS patterns
PATTERNS = {
    'Aromatic ring':      'c1ccccc1',
    'Carboxylic acid':    'C(=O)[OH]',
    'Ester':              'C(=O)OC',
    'Amide':              'C(=O)N',
    'Amine':              '[NH2]',
    'Hydroxyl':           '[OH]',
    'Halogen':            '[F,Cl,Br,I]',
    'Carbonyl':           'C=O',
    'Nitrogen heterocycle': 'n1ccccc1',
    'Sulfone':            'S(=O)(=O)',
}

def count_matches(smiles_list, smarts):
    patt = Chem.MolFromSmarts(smarts)
    count = 0
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol and mol.HasSubstructMatch(patt):
            count += 1
    return count

high_smiles = [dataset_scaled[i].smiles for i in high_idx]
low_smiles  = [dataset_scaled[i].smiles for i in low_idx]

pattern_data = []
for name, smarts in PATTERNS.items():
    n_high = count_matches(high_smiles, smarts)
    n_low  = count_matches(low_smiles,  smarts)
    frac_high = n_high / len(high_smiles) if high_smiles else 0
    frac_low  = n_low  / len(low_smiles)  if low_smiles  else 0
    enrichment = frac_high / (frac_low + 1e-6)
    pattern_data.append({
        'Pattern': name,
        'Freq_High': frac_high,
        'Freq_Low':  frac_low,
        'Enrichment': enrichment,
    })

pattern_df = pd.DataFrame(pattern_data).sort_values('Enrichment', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(pattern_df))
w = 0.35
ax.bar(x - w/2, pattern_df['Freq_High'], w, label='High Bias Score (top 25%)',
       color='steelblue', edgecolor='k', alpha=0.85)
ax.bar(x + w/2, pattern_df['Freq_Low'],  w, label='Low Bias Score (bottom 25%)',
       color='salmon', edgecolor='k', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(pattern_df['Pattern'], rotation=25, ha='right', fontsize=9)
ax.set_ylabel('Fraction of Molecules with Pattern')
ax.set_title('Functional Group Frequency: High vs Low Bias Score Molecules', fontsize=12)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('substructure_enrichment.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFunctional group enrichment analysis:")
print(pattern_df.round(3).to_string(index=False))

## 8. — Comprehensive XAI Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(18, 14))
gs  = fig.add_gridspec(3, 3, hspace=0.45, wspace=0.35)

# ─── Panel 1: CV Distribution ─────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(cv_arr, bins=12, color='orange', edgecolor='k', alpha=0.85)
ax1.axvline(cv_arr.mean(), color='red', linestyle='--',
            label=f'μ={cv_arr.mean():.3f}')
ax1.set_title('GNNExplainer CV\n(Stochasticity)', fontsize=10)
ax1.set_xlabel('CV'); ax1.legend(fontsize=8)

# ─── Panel 2: GNN Atom Importance ────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
top6 = sorted(atom_means_gnn, key=atom_means_gnn.get, reverse=True)[:6]
ax2.barh(top6[::-1], [atom_means_gnn[a] for a in top6[::-1]],
         color='mediumseagreen', edgecolor='k', alpha=0.85)
ax2.set_title('GNNExplainer\nTop Atom Types', fontsize=10)
ax2.set_xlabel('Importance')

# ─── Panel 3: IG Feature Importance ─────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
top8_ig = sorted_idx[:8]
ax3.barh([feature_names[i] for i in top8_ig[::-1]],
         [ig_feature_importance[i] for i in top8_ig[::-1]],
         color='steelblue', edgecolor='k', alpha=0.85)
ax3.set_title('Integrated Gradients\nTop Node Features', fontsize=10)
ax3.set_xlabel('IG Attribution')

# ─── Panel 4: Bias Score Distribution ────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
ax4.hist(bias_arr, bins=12, color='purple', edgecolor='k', alpha=0.85)
ax4.axvline(np.percentile(bias_arr, 75), color='green', linestyle='--',
            label=f'Q75={np.percentile(bias_arr,75):.2f}')
ax4.axvline(np.percentile(bias_arr, 25), color='red', linestyle='--',
            label=f'Q25={np.percentile(bias_arr,25):.2f}')
ax4.set_title('Bias Score Distribution', fontsize=10)
ax4.set_xlabel('Bias Score'); ax4.legend(fontsize=8)

# ─── Panel 5: GU vs cAMP colored by Bias ────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
sc5 = ax5.scatter(all_preds_orig[:,0], all_preds_orig[:,1],
                  c=bias_arr, cmap='RdYlGn', s=60, edgecolors='k', linewidths=0.3)
plt.colorbar(sc5, ax=ax5, label='Bias')
ax5.set_xlabel('Predicted GU (↑)'); ax5.set_ylabel('Predicted cAMP (↓)')
ax5.set_title('GU vs cAMP\nBias Score', fontsize=10)

# ─── Panel 6: Substructure Enrichment ────────────────────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
top5_patt = pattern_df.head(5)
x6 = np.arange(len(top5_patt))
ax6.bar(x6 - 0.2, top5_patt['Freq_High'], 0.38, label='High', color='steelblue', alpha=0.85)
ax6.bar(x6 + 0.2, top5_patt['Freq_Low'],  0.38, label='Low',  color='salmon',    alpha=0.85)
ax6.set_xticks(x6)
ax6.set_xticklabels(top5_patt['Pattern'], rotation=20, ha='right', fontsize=8)
ax6.set_title('Substructure Enrichment\n(High vs Low Bias)', fontsize=10)
ax6.legend(fontsize=8)

# ─── Panel 7: OOF GU ─────────────────────────────────────────────────────────
ax7 = fig.add_subplot(gs[2, 0])
ax7.scatter(all_true_orig[:,0], all_preds_orig[:,0], c='steelblue', alpha=0.7, s=40)
lims = [all_true_orig[:,0].min(), all_true_orig[:,0].max()]
ax7.plot(lims, lims, 'k--', alpha=0.5)
r2gu = r2_score(all_true_orig[:,0], all_preds_orig[:,0])
ax7.set_title(f'OOF: GU\nR²={r2gu:.3f}', fontsize=10)
ax7.set_xlabel('True'); ax7.set_ylabel('Predicted')

# ─── Panel 8: OOF cAMP ───────────────────────────────────────────────────────
ax8 = fig.add_subplot(gs[2, 1])
ax8.scatter(all_true_orig[:,1], all_preds_orig[:,1], c='coral', alpha=0.7, s=40)
lims2 = [all_true_orig[:,1].min(), all_true_orig[:,1].max()]
ax8.plot(lims2, lims2, 'k--', alpha=0.5)
r2camp = r2_score(all_true_orig[:,1], all_preds_orig[:,1])
ax8.set_title(f'OOF: cAMP\nR²={r2camp:.3f}', fontsize=10)
ax8.set_xlabel('True'); ax8.set_ylabel('Predicted')

# ─── Panel 9: CV vs Bias ─────────────────────────────────────────────────────
ax9 = fig.add_subplot(gs[2, 2])
ax9.scatter(bias_arr, cv_arr, c='purple', alpha=0.7, edgecolors='k', linewidths=0.3)
ax9.set_xlabel('Bias Score')
ax9.set_ylabel('GNNExplainer CV')
ax9.set_title(f'Stability vs Desirability\nr={corr:.3f}', fontsize=10)

fig.suptitle(
    'B2AR Multi-Output MPNN — Complete XAI Dashboard\n'
    'Objective: High Glucose Uptake (GU) & Low cAMP',
    fontsize=14, fontweight='bold', y=1.01
)
plt.savefig('xai_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("Dashboard saved to xai_dashboard.png")

### 8.1 — Final Results Summary Table

In [ ]:
print("=" * 65)
print(" B2AR Multi-Output MPNN — Complete Results Summary")
print("=" * 65)

# CV metrics
print("\n── 5-Fold Cross-Validation ─────────────────────────────────")
print(f"  GU   R²  :  {summary.loc['mean','R2_GU']:.3f} ± {summary.loc['std','R2_GU']:.3f}")
print(f"  GU   MAE :  {summary.loc['mean','MAE_GU']:.3f} ± {summary.loc['std','MAE_GU']:.3f}")
print(f"  GU   RMSE:  {summary.loc['mean','RMSE_GU']:.3f} ± {summary.loc['std','RMSE_GU']:.3f}")
print(f"  cAMP R²  :  {summary.loc['mean','R2_cAMP']:.3f} ± {summary.loc['std','R2_cAMP']:.3f}")
print(f"  cAMP MAE :  {summary.loc['mean','MAE_cAMP']:.3f} ± {summary.loc['std','MAE_cAMP']:.3f}")
print(f"  cAMP RMSE:  {summary.loc['mean','RMSE_cAMP']:.3f} ± {summary.loc['std','RMSE_cAMP']:.3f}")

# Bias score
print("\n── Bias Score (High-GU & Low-cAMP) ────────────────────────")
print(f"  Mean BiasScore: {bias_arr.mean():.3f} ± {bias_arr.std():.3f}")
top5_names = results_df['Molecule Name'].head(5).tolist()
top5_bs    = results_df['BiasScore_pred'].head(5).tolist()
print(f"  Top-5 molecules: {list(zip(top5_names, [f'{b:.3f}' for b in top5_bs]))}")

# XAI
print("\n── GNNExplainer ────────────────────────────────────────────")
top3_atom_gnn = sorted(atom_means_gnn, key=atom_means_gnn.get, reverse=True)[:3]
print(f"  Top-3 atom types: {top3_atom_gnn}")
print(f"  Mean stochasticity (CV): {cv_arr.mean():.4f} ± {cv_arr.std():.4f}")
print(f"  Runs for stochasticity correction: {N_RUNS}")

print("\n── Integrated Gradients ───────────────────────────────────")
top3_feat_ig = [feature_names[i] for i in sorted_idx[:3]]
print(f"  Top-3 node features: {top3_feat_ig}")

print("\n── Most Enriched Substructures (High Bias Molecules) ──────")
for _, row in pattern_df.head(3).iterrows():
    print(f"  {row['Pattern']:25s}: enrichment={row['Enrichment']:.2f}x")

print("\n" + "=" * 65)

## 9 — Save Results

In [ ]:
# Save molecule results with bias scores
results_df.to_csv('b2ar_results.csv', index=False)

# Save metrics
metrics_df.to_csv('cv_metrics.csv')

# Save IG attributions as numpy
np.save('ig_attributions.npy', np.array(ig_attributions_all, dtype=object), allow_pickle=True)

# Save GNNExplainer importances
gnn_summary = pd.DataFrame({
    'Molecule': [dataset_scaled[i].name for i in range(len(dataset_scaled))],
    'mean_node_importance': [m.mean() for m in gnn_node_imps_mean],
    'cv_stochasticity': cv_arr,
    'bias_score': bias_arr,
})
gnn_summary.to_csv('gnnexplainer_summary.csv', index=False)

# Save final model
torch.save(final_model.state_dict(), 'b2ar_mpnn_final.pt')

print("Saved files:")
for f in ['b2ar_results.csv', 'cv_metrics.csv', 'ig_attributions.npy',
          'gnnexplainer_summary.csv', 'b2ar_mpnn_final.pt']:
    size = os.path.getsize(f) if os.path.exists(f) else 0
    print(f"  {f}  ({size:,} bytes)")

---

## Summary & Interpretation Guide

### Model Architecture
The **Multi-Output MPNN** uses NNConv (edge-conditioned message passing) with two separate output heads (GU head + cAMP head) that share a common molecular representation. This allows the model to learn joint molecular structure–activity relationships for both targets simultaneously.

### Bias Score
$$\text{BiasScore} = \frac{\hat{GU}_{norm} + (1 - \hat{cAMP}_{norm})}{2}$$
Molecules with BiasScore → 1 are predicted to have both **high glucose uptake** and **low cAMP**, representing the ideal B2AR modulator profile.

### GNNExplainer
- Run **N times** per molecule to average out stochasticity; report mean ± std
- Low CV = stable explanation = more reliable atom importance
- Atom-type importances reveal **which elements** drive the combined objective

### Integrated Gradients
- Deterministic, gradient-based attribution at the **feature level**
- Identifies which atom properties (aromaticity, hybridization, charge) matter most
- Complementary to GNNExplainer's graph-structure perspective

### How to Use Results on Real Data
1. Replace the synthetic SMILES with your actual B2AR dataset
2. The high-scoring molecules (top BiasScore) are candidates for synthesis/testing
3. Cross-reference GNNExplainer atom-type rankings with IG feature rankings for consensus
4. Enriched functional groups in high-scoring molecules guide medicinal chemistry hypotheses